In [1]:
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadTestingIO\BoSSSpad.dll"
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadTestingIOAfter\BoSSSpad.dll"
#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadTestingIOAfterMPI\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Text;
using System.Globalization;
using System.Threading;
using System.Threading.Tasks;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using ZwoLevelSetSolver;
using ZwoLevelSetSolver.SolidPhase;
Init();

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
//ExecutionQueues

In [3]:
//GetDefaultQueue()

In [4]:
static var myBatch = ExecutionQueues[2];
static var myDb = myBatch.CreateOrOpenCompatibleDatabase("Droplet-EE-LikeEL-TestingIO");

In [5]:
BoSSSshell.WorkflowMgm.Init("Droplet-EE-LikeEL-TestingIO");

Project name is set to 'Droplet-EE-LikeEL-TestingIO'.
Default Execution queue is chosen for the database.
Opening existing database '\\dc3\userspace\miao\cluster\Droplet-EE-LikeEL-TestingIO'.


In [6]:
BoSSSshell.WorkflowMgm.DefaultDatabase = myDb;

## Create grid

In [7]:
public static class GridFactory {
 
    public static Grid2D GenerateGrid() { 
        double xSize = 2;
        double yTop = 1.5 - 20.0 / 176.7;
        double yBottom = -20.0 / 176.7;
        //int kelem = 8;
        int kelem = 24;

        double[] Xnodes = GenericBlas.Linspace(-xSize, xSize, kelem + 1);
        double[] Ynodes = GenericBlas.Linspace(yBottom, yTop, kelem / 8 * 3 + 1);
                var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes);

                grd.EdgeTagNames.Add(1, "wall_lower");
                grd.EdgeTagNames.Add(2, "pressure_outlet_upper");
                //grd.EdgeTagNames.Add(3, "wall_left");
                //grd.EdgeTagNames.Add(4, "wall_right");
                grd.EdgeTagNames.Add(3, "FreeSlip_left");
                grd.EdgeTagNames.Add(4, "FreeSlip_right");

                grd.DefineEdgeTags(delegate (double[] X) {
                    byte et = 0;
                    if(Math.Abs(X[1] - yBottom) <= 1.0e-8)
                        et = 1;
                    if(Math.Abs(X[1] - yTop) <= 1.0e-8)
                        et = 2;
                    if(Math.Abs(X[0] + xSize) <= 1.0e-8)
                        et = 3;
                    if(Math.Abs(X[0] - xSize) <= 1.0e-8)
                        et = 4;

                    return et;
                });

                return grd;
     }
 
 }

In [8]:
public static class BoundaryValueFactory { 
    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
           
           stw.WriteLine("static class BoundaryValues {");
           stw.WriteLine("  static public double Zero(double[] X) {");
           stw.WriteLine("    return 0.0;");
           stw.WriteLine("  }");

           stw.WriteLine("  static public double pJump(double[] X) {");
           stw.WriteLine("    return 2 * 0.046 / 0.4;");
           stw.WriteLine("  }");
           
           stw.WriteLine(" static public double InitialPhi(double[] X) { ");
           //stw.WriteLine("    return ((X[0] - 0.0).Pow2() + (X[1] - 0.0).Pow2() - 0.16);");
            
           stw.WriteLine("    if(X[1] >= (0)) {");
           stw.WriteLine("    return ((X[0] - 0.0).Pow2() + (X[1] - 0.0).Pow2() - 0.16);");
           stw.WriteLine("    }");
           stw.WriteLine("    return ((X[0] - 0.0).Pow2() + (0.0 - 0.0).Pow2() - 0.16);");
            
           stw.WriteLine("    }"); 
           
           stw.WriteLine(" static public double InitialPhi1(double[] X) { ");
           stw.WriteLine("    return -(X[1] - 0);");
           stw.WriteLine("    }"); 

           stw.WriteLine("}"); 
           return stw.ToString();
        }
    }
   
    static public Formula Get_Zero() {
        return new Formula("BoundaryValues.Zero", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_pJump(){
        return new Formula("BoundaryValues.pJump", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InitialPhi(){
        return new Formula("BoundaryValues.InitialPhi", AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_InitialPhi1(){
        return new Formula("BoundaryValues.InitialPhi1", AdditionalPrefixCode:GetPrefixCode());
    }
}

## Create Control file

In [9]:
        public static ZLS_Control Aland( int p = 2, int AMRlvl = 0, int MpiNum = 1) {
            ZLS_Control C = new ZLS_Control(p);
            //C.ImmediatePlotPeriod = 1;
            //C.SuperSampling = 3;

            C.AgglomerationThreshold = 0.3;
            C.NoOfMultigridLevels = 1;

            //int D = 2;

            AppControl._TimesteppingMode compMode = AppControl._TimesteppingMode.Transient;

            //_DbPath = @"\\fdyprime\userspace\smuda\cluster\cluster_db";
            //_DbPath = @"D:\local\local_Testcase_databases\Testcase_ContactLine";
            //_DbPath = @"D:\local\local_spatialConvStudy\StaticDropletOnPlateConvergence\SDoPConvDB";

            // basic database options
            // ======================
            #region db

            C.savetodb = true;
            //C.DbPath = @"\\dc1\userspace\miao\cluster\Droplet-EE-LikeEL-DT";
            C.ProjectName = "Droplet";
            //C.SessionName = "Droplet-LikeEL-AMR" + AMRlvl + "-Merged-MPI" + MpiNum + "-TestingIO";
            C.SessionName = "Droplet-LikeEL-AMR" + AMRlvl + "-Merged-MPI" + MpiNum + "-AMRMOD";
            C.ProjectDescription = "Droplet running on pc";

            C.ContinueOnIoError = false;

            //C.LogValues = XNSE_Control.LoggingValues.MovingContactLine;
            //C.PostprocessingModules.Add(new MovingContactLineLogging());

            #endregion

            // Physical Parameters
            // ===================
            #region physics

            double scale = 4.4175e-4; // For a droplet with radius r = 176.7 micrometre
            C.PhysicalParameters.rho_A = 10 * scale * scale * scale;
            C.PhysicalParameters.rho_B = 1260 * scale * scale * scale;
            C.PhysicalParameters.mu_A = 0.1 * scale ;
            C.PhysicalParameters.mu_B = 1.41 * scale;
            double sigma = 0.046;
            C.PhysicalParameters.Sigma = sigma;

            C.PhysicalParameters.betaS_A = 0.0;
            C.PhysicalParameters.betaS_B = 0.0;

            C.PhysicalParameters.betaL = 0.0;
            C.PhysicalParameters.theta_e = Math.PI / 2.0;

            C.PhysicalParameters.IncludeConvection = true;
            C.PhysicalParameters.Material = false; //??

            C.Material = new Solid() {
                Density = 1000 * scale * scale * scale,
                Lame2 = 1000 * scale,
                Viscosity = 1 * scale, // Real Viscosity
            };

            #endregion

            // grid generation
            // ===============
            #region grid

            C.SetGrid(GridFactory.GenerateGrid());

            #endregion

            // Initial Values
            // ==============

            C.AddInitialValue("Pressure#A", BoundaryValueFactory.Get_pJump());
            C.AddInitialValue("Pressure#B", BoundaryValueFactory.Get_Zero());
            C.AddInitialValue("Phi", BoundaryValueFactory.Get_InitialPhi());
            C.AddInitialValue("Phi2", BoundaryValueFactory.Get_InitialPhi1());
            //C.AddInitialValue(ZwoLevelSetSolver.VariableNames.SolidLevelSetCG, BoundaryValueFactory.Get_InitialPhi1());



            // boundary conditions
            // ===================
            #region BC


            C.AddBoundaryValue("wall_lower");
            C.AddBoundaryValue("pressure_outlet_upper");
            //C.AddBoundaryValue("wall_left");
            //C.AddBoundaryValue("wall_right");
            C.AddBoundaryValue("FreeSlip_left");
            C.AddBoundaryValue("FreeSlip_right");

            C.AdvancedDiscretizationOptions.GNBC_Localization = NavierSlip_Localization.Bulk;
            C.AdvancedDiscretizationOptions.GNBC_SlipLength = NavierSlip_SlipLength.Prescribed_Beta;
            //C.PhysicalParameters.sliplength = 0.001;

            #endregion

            // misc. solver options
            // ====================
            #region solver


            //C.AdvancedDiscretizationOptions.CellAgglomerationThreshold = 0.2;
            //C.AdvancedDiscretizationOptions.PenaltySafety = 40;
            //C.AdvancedDiscretizationOptions.UseGhostPenalties = true;

            C.NonLinearSolver.MaxSolverIterations = 160;
            C.NonLinearSolver.MinSolverIterations = 2;
            //C.Solver_MaxIterations = 50;
            C.NonLinearSolver.ConvergenceCriterion = 1e-8;
            //C.Solver_ConvergenceCriterion = 1e-8;
            C.LevelSet_ConvergenceCriterion = 1e-12;
            C.NonLinearSolver.Globalization = BoSSS.Solution.AdvancedSolvers.Newton.GlobalizationOption.Dogleg;


            //C.Option_LevelSetEvolution = (compMode == AppControl._TimesteppingMode.Steady) ? LevelSetEvolution.None : LevelSetEvolution.FastMarching;
            //C.EllipticExtVelAlgoControl.solverFactory = () => new ilPSP.LinSolvers.PARDISO.PARDISOSolver();
            //C.EllipticExtVelAlgoControl.IsotropicViscosity = 1e-3;
            //C.fullReInit = false; 

            C.AdvancedDiscretizationOptions.FilterConfiguration = CurvatureAlgorithms.FilterConfiguration.NoFilter;
            C.AdvancedDiscretizationOptions.ViscosityMode = ViscosityMode.FullySymmetric;
            C.AdvancedDiscretizationOptions.SST_isotropicMode = BoSSS.Solution.XNSECommon.SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;

            C.AdaptiveMeshRefinement = true;
            C.AMR_startUpSweeps = AMRlvl;
            //C.activeAMRlevelIndicators.Add(new ContactPointRefiner { maxRefinementLevel = AMRlvl});
            C.activeAMRlevelIndicators.Add(new AMRatInnerContactLine { maxRefinementLevel = AMRlvl, levelSets = new[] { 0, 1 }, FieldWidth = 0 });
            //C.activeAMRlevelIndicators.Add(new AMRatInnerContactLine { maxRefinementLevel = AMRlvl, levelSets = new[], FieldWidth = 1 });
            //C.activeAMRlevelIndicators.Add(new AMRonNarrowband { maxRefinementLevel = AMRlvl, levelSet = 0 });

            #endregion
                
            C.DynamicLoadBalancing_On = true;
            C.DynamicLoadBalancing_Period = 20;
            C.DynamicLoadBalancing_RedistributeAtStartup = true;
            C.GridPartType = GridPartType.METIS;


            // Timestepping
            // ============
            #region time

            //C.CheckJumpConditions = true;

            C.TimeSteppingScheme = TimeSteppingScheme.BDF2;
            C.Timestepper_BDFinit = TimeStepperInit.SingleInit;
            C.Timestepper_LevelSetHandling = LevelSetHandling.LieSplitting;
            

            C.TimesteppingMode = compMode;
            //double dt = 5e-7;
            double dt = 2e-5;
            C.dtMax = dt;
            C.dtMin = dt;
            C.Endtime = 100;
            C.NoOfTimesteps = 200;
            C.saveperiod = 1;

            C.TracingNamespaces = "*";
            
            #endregion
            
            return C;
        }

## Send and run jobs

In [10]:
int[] MpiNUMs = new int[] {4};
//int[] MpiNUMs = new int[] {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12};

In [11]:
foreach(int MpiNUM in MpiNUMs){
    var C_CTRLFILE = Aland(3, 5, MpiNUM);//Create control file.
    var JobLocal = C_CTRLFILE.CreateJob();
    JobLocal.NumberOfMPIProcs = MpiNUM;
    JobLocal.NumberOfThreads = 1;
    JobLocal.Activate(myBatch);
    JobLocal.ShowOutput();
    }

Grid Edge Tags changed.
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job Droplet-LikeEL-AMR5-Merged-MPI4-AMRMOD ... 
Opening existing database 'C:\Users\miao\Documents\Database\Droplet-EE-LikeEL-TestingIO'.
Set Database: { Session Count = 83; Grid Count = 334; Path = \\dc3\userspace\miao\cluster\Droplet-EE-LikeEL-TestingIO }
Grid is not in database yet...
Grid successfully saved: 6002ed5a-8dc5-4855-9e18-b0e9f88a5e2d


Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\miao\cluster\Deployment\Droplet-EE-LikeEL-TestingIO-ZwoLevelSetSolver2024May29_142216.530092
copied 46 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.

Starting external console ...
(You may close the new window at any time, the job will continue.)


In [12]:
wmg.DefaultDatabase.Sessions

#0: Droplet-EE-LikeEL-TestingIO	Droplet-LikeEL-AMR5-Merged-MPI4-AMRMOD*	05/29/2024 14:25:24	20276e84...
#1: Droplet-EE-LikeEL-TestingIO	Droplet-LikeEL-AMR5-Merged-MPI8-AMRMOD	05/29/2024 09:47:45	3868ae94...
#2: Droplet-EE-LikeEL-TestingIO	Droplet-LikeEL-AMR5-Merged-MPI6-AMRMOD	05/29/2024 09:47:12	3f9893a9...
#3: Droplet-EE-LikeEL-TestingIO	Droplet-LikeEL-AMR5-Merged-MPI7-AMRMOD	05/29/2024 09:46:15	55b776f4...
#4: Droplet-EE-LikeEL-TestingIO	Droplet-LikeEL-AMR5-Merged-MPI5-AMRMOD	05/29/2024 09:45:22	741c6dcf...
#5: Droplet-EE-LikeEL-TestingIO	Droplet-LikeEL-AMR5-Merged-MPI4-AMRMOD	05/29/2024 09:42:49	768bdd44...
#6: Droplet-EE-LikeEL-TestingIO	Droplet-LikeEL-AMR5-Merged-MPI2-AMRMOD	05/29/2024 09:40:53	8177edfc...
#7: Droplet-EE-LikeEL-TestingIO	Droplet-LikeEL-AMR5-Merged-MPI3-AMRMOD	05/29/2024 09:41:38	0bf28406...
#8: Droplet-EE-LikeEL-TestingIO	Droplet-LikeEL-AMR5-Merged-MPI1-AMRMOD	05/29/2024 09:39:59	78694acb...
#9: Droplet-EE-LikeEL-TestingIO	Droplet-LikeEL-AMR5-Merged-MPI9-AMRMOD	0

In [13]:
wmg.DefaultDatabase.Sessions[0].Timesteps.Count

21

In [14]:
//var outPath = wmg.DefaultDatabase.Sessions[0].Timesteps.Every(1).Skip(0).Export().WithSupersampling(3).Do();

Starting export process... Data will be written to the directory: C:\Users\miao\AppData\Local\BoSSS\plots\sessions\Droplet-EE-LikeEL-TestingIO__Droplet-LikeEL-AMR5-Merged-MPI4-AMRMOD__20276e84-1d41-4fa2-a181-bb20c5421077


In [13]:
for (int i = 0; i < 8; i++){
    var outPath = wmg.DefaultDatabase.Sessions[i].Timesteps.Every(1).Skip(0).Export().WithSupersampling(3).Do();
}

Starting export process... Data will be written to the directory: C:\Users\miao\AppData\Local\BoSSS\plots\sessions\Droplet-EE-LikeEL-TestingIO__Droplet-LikeEL-AMR5-Merged-MPI6-AMRMOD__f307a8d7-f26b-48e0-8ed6-aa7ae82ee77c
Starting export process... Data will be written to the directory: C:\Users\miao\AppData\Local\BoSSS\plots\sessions\Droplet-EE-LikeEL-TestingIO__Droplet-LikeEL-AMR5-Merged-MPI5-AMRMOD__c7d03d56-7a24-4937-bda5-3e314245f93f
Starting export process... Data will be written to the directory: C:\Users\miao\AppData\Local\BoSSS\plots\sessions\Droplet-EE-LikeEL-TestingIO__Droplet-LikeEL-AMR5-Merged-MPI4-AMRMOD__d19b6a73-e399-4eae-b724-5ac5e4b7eec3
Starting export process... Data will be written to the directory: C:\Users\miao\AppData\Local\BoSSS\plots\sessions\Droplet-EE-LikeEL-TestingIO__Droplet-LikeEL-AMR5-Merged-MPI3-AMRMOD__bb38227e-e0f6-4d95-bc2f-a75c31964e6d
Starting export process... Data will be written to the directory: C:\Users\miao\AppData\Local\BoSSS\plots\sessions\D